In [16]:
from datetime import timedelta

from snowflake.snowpark import Session, context
import json

In [17]:
# session = Session.builder.config("connection_name", "default").create()
session = context.get_active_session()

In [18]:
session.sql("SHOW TASKS;").show()

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"created_on"                      |"name"                     |"id"                                  |"database_name"  |"schema_name"  |"owner"       |"comment"  |"warehouse"  |"schedule"    |"predecessors"                                     |"state"    |"definition"                                        |"condition"  |"allow_overlapping_execution"  |"error_integ

In [19]:
folder = r'workflows/'

In [20]:
with open(folder+'mainworkflow.json') as f:
    print(type(f))
    d = json.load(f)
    print(d)

<class '_io.TextIOWrapper'>
{'wf_NUODATA_TEST': {'stepType': 'mainWF', 'depends_on': 'None', 'notebook': 'None'}, 'Wkl_NUODATA_TEST_Pre_Load': {'stepType': 'Sub Workflow', 'depends_on': 'wf_NUODATA_TEST', 'notebook': '..../WorkflowExecutor'}, 'Wkl_NUODATA_TEST_BASE_Load': {'stepType': 'Sub Workflow', 'depends_on': 'Wkl_NUODATA_TEST_Pre_Load', 'notebook': '..../WorkflowExecutor'}, 's_Last_CALENDER_DATE': {'stepType': 'session', 'depends_on': 'Wkl_NUODATA_TEST_BASE_Load', 'notebook': '..../s_Last_CALENDER_DATE'}}


In [21]:
# file_opner("mainworkflow.json", "workflows")

Using SP to call mappings notebook


In [22]:
session.sql("""
    CREATE OR REPLACE PROCEDURE EXECUTE_NOTEBOOK(NOTEBOOK_NAME STRING)
    RETURNS STRING
    LANGUAGE SQL
    AS
    $$
    BEGIN
        EXECUTE IMMEDIATE 'EXECUTE NOTEBOOK ' || NOTEBOOK_NAME || '();';
        RETURN 'Notebook ' || NOTEBOOK_NAME || ' executed successfully!';
    END;
    $$;
""").collect()

[Row(status='Function EXECUTE_NOTEBOOK successfully created.')]

Using SP to call the subworkflow task

In [23]:
session.sql("""
    CREATE OR REPLACE PROCEDURE Call_Sub_Task_Graph(TASK_NAME STRING)
    RETURNS STRING
    LANGUAGE SQL
    AS
    $$
    BEGIN
        EXECUTE IMMEDIATE 'EXECUTE TASK ' || TASK_NAME ||';';
        RETURN 'TASK ' || TASK_NAME || ' executed successfully!';
    END;
    $$;
    """).collect()


[Row(status='Function CALL_SUB_TASK_GRAPH successfully created.')]

In [24]:
data = {
    "wf_NUODATA_TEST":{"workflowType":"mainWF","stepType": "None", "depends_on": "None", "notebook": "None"},
    "Wkl_NUODATA_TEST_Pre_Load":{"stepType": "Sub Workflow", "depends_on": "wf_NUODATA_TEST", "notebook": "..../WorkflowExecutor"},
    "Wkl_NUODATA_TEST_BASE_Load":{"stepType": "Sub Workflow", "depends_on": "Wkl_NUODATA_TEST_Pre_Load", "notebook": "..../WorkflowExecutor"},
    "s_Last_CALENDER_DATE":{"stepType": "session", "depends_on": "Wkl_NUODATA_TEST_BASE_Load", "notebook": "..../s_Last_CALENDER_DATE"},
}

In [25]:
database = "DELTA_TRAINING"
schema = "PUBLIC"
warehouse = "COMPUTE_WH"

In [26]:
from contextlib import contextmanager
from snowflake.snowpark.exceptions import SnowparkSQLException

@contextmanager
def snowflake_exception_handler():
    try:
        yield
    except SnowparkSQLException as e:
        print(f"\033[91mError executing Snowflake query: {e}\033[0m")

In [ ]:
# class TaskGraph:
#     database = "DAGs"
#     schema = "PUBLIC"
#     warehouse = "COMPUTE_WH"
#     folder = "workflows"
#     def __init__(self, folder):
#         self.folder = folder
#         self.counter = 0

#     def start_create(self, main_workflow_file):
#         self.file_opner(main_workflow_file)

#     def parser(self, file_data:dict):
#         for task_name in file_data:
#             self.create_task(task_name, file_data)

#     def file_opner(self, file_name:str):
#         folder = self.folder + "/" if self.folder!="" else self.folder
#         first_child = None
#         with open(folder + file_name + ".json") as f:
#             file_data = json.load(f)
#             first_child = next(iter(file_data))
#             self.parser(file_data)
#         return first_child
        

#     def create_task(self, task_name, data):
#         self.counter+=1
#         depends_on = data[task_name]['depends_on']
#         notebook = data[task_name]['notebook']
#         if  '..../' in notebook:
#             notebook = notebook.removeprefix('..../')
#         print(f"Task {self.counter} : {task_name} depends on {depends_on}")

#         # if depends_on!='None' and data.get(depends_on).get('stepType') == "Sub Workflow":
#         #         depends_on = depends_on + "_caller"
        
#         if data[task_name]["stepType"] == "Sub Workflow":
#             with snowflake_exception_handler():
#                 first_child = self.file_opner(file_name=task_name)
#                 print(f"CREATING SUB WORKFLOW: {task_name} which calls {first_child} and depends on {depends_on}")
#                 if depends_on == 'None':
#                     session.sql(
#                     f"""
#                         CREATE OR REPLACE TASK {database}.{schema}.{task_name}
#                         WAREHOUSE = {warehouse}
#                         schedule='999 minute'
#                         AS
#                         CALL CALL_SUB_TASK_GRAPH('{first_child}')
#                     """).collect() # we can call stored proc here as well, it shall call the same name as the workflow

#                 else:
#                     session.sql(
#                     f"""
#                         CREATE OR REPLACE TASK {database}.{schema}.{task_name}
#                         WAREHOUSE = {warehouse}
#                         AFTER {depends_on}
#                         AS
#                         CALL CALL_SUB_TASK_GRAPH('{first_child}')
#                     """).collect() # we can call stored proc here as well, it shall call the same name as the workflow
                    
#                 print(f"Created SUB WORKFLOW: {task_name} which depends on {depends_on}")
        
#         elif data[task_name]["stepType"] == "session":
#             with snowflake_exception_handler():
#                 print(f"CREATING session : {task_name} which depends on {depends_on}")
#                 if depends_on == 'None':
#                     session.sql(
#                     f"""
#                         CREATE OR REPLACE TASK {database}.{schema}.{task_name}
#                         WAREHOUSE = {warehouse}
#                         schedule='999 minute'
#                         AS
#                         CALL EXECUTE_NOTEBOOK('{notebook}');
#                     """).collect()

#                 else:
#                     session.sql(
#                     f"""
#                         CREATE OR REPLACE TASK {database}.{schema}.{task_name}
#                         WAREHOUSE = {warehouse}
#                         AFTER {depends_on}
#                         AS
#                         CALL EXECUTE_NOTEBOOK('{notebook}');
#                     """).collect()
#                 print(task_name, " notebook task created successfully")

#         elif depends_on == "None":
#             with snowflake_exception_handler():
#                 print(f"CREATING Main WORKFLOW: {task_name} which depends on {depends_on}")
#                 session.sql(
#                     f"""create or replace task {database}.{schema}.{task_name}
#                     warehouse={warehouse}
#                     schedule='999 minute'
#                     as SELECT 1;
#                     """).collect()
#                 print(task_name, " root task is successfully created")
            
        

In [ ]:
class TaskGraph:

    def __init__(self, folder):
        self.folder = folder
        self.counter = 0

    def start_create(self, main_workflow_file):
        self.file_opner(main_workflow_file)

    def parser(self, file_data:dict):
        for task_name in file_data:
            self.create_task(task_name, file_data)

    def file_opner(self, file_name:str):
        folder = self.folder + "/" if self.folder!="" else self.folder
        first_child = None
        with open(folder + file_name + ".json") as f:
            file_data = json.load(f)
            first_child = next(iter(file_data))
            self.parser(file_data)
        return first_child
        

    def create_task(self, task_name, data):
        self.counter+=1
        depends_on = data[task_name]['depends_on']
        notebook = data[task_name]['notebook']
        if  '..../' in notebook:
            notebook = notebook.removeprefix('..../')
        print(f"Task {self.counter} : {task_name} depends on {depends_on}")

        # if depends_on!='None' and data.get(depends_on).get('stepType') == "Sub Workflow":
        #         depends_on = depends_on + "_caller"
        
        if data[task_name]["stepType"] == "Sub Workflow":
            with snowflake_exception_handler():
                first_child = self.file_opner(file_name=task_name)
                print(f"CREATING SUB WORKFLOW: {task_name} which calls {first_child} and depends on {depends_on}")
                if depends_on == 'None':
                    session.sql(
                    f"""
                        CREATE OR REPLACE TASK {database}.{schema}.{task_name}
                        WAREHOUSE = {warehouse}
                        AFTER {depends_on}
                        AS
                        CALL CALL_SUB_TASK_GRAPH('{first_child}')
                    """).collect() # we can call stored proc here as well, it shall call the same name as the workflow

                else:
                    session.sql(
                    f"""
                        CREATE OR REPLACE TASK {database}.{schema}.{task_name}
                        WAREHOUSE = {warehouse}
                        AFTER {depends_on}
                        AS
                        CALL CALL_SUB_TASK_GRAPH('{first_child}')
                    """).collect() # we can call stored proc here as well, it shall call the same name as the workflow
                    
                print(f"Created SUB WORKFLOW: {task_name} which depends on {depends_on}")
        
        elif data[task_name]["stepType"] == "session":
            with snowflake_exception_handler():
                print(f"CREATING session : {task_name} which depends on {depends_on}")
                if depends_on == 'None':
                    session.sql(
                    f"""
                        CREATE OR REPLACE TASK {database}.{schema}.{task_name}
                        WAREHOUSE = {warehouse}
                        schedule='999 minute'
                        AS
                        CALL EXECUTE_NOTEBOOK('{notebook}');
                    """).collect()

                else:
                    session.sql(
                    f"""
                        CREATE OR REPLACE TASK {database}.{schema}.{task_name}
                        WAREHOUSE = {warehouse}
                        AFTER {depends_on}
                        AS
                        CALL EXECUTE_NOTEBOOK('{notebook}');
                    """).collect()
                print(task_name, " notebook task created successfully")

        elif depends_on == "None":
            with snowflake_exception_handler():
                print(f"CREATING Main WORKFLOW: {task_name} which depends on {depends_on}")
                session.sql(
                    f"""create or replace task {database}.{schema}.{task_name}
                    warehouse={warehouse}
                    schedule='999 minute'
                    as SELECT 1;
                    """).collect()
                print(task_name, " root task is successfully created")
            
        

In [65]:
a = TaskGraph("workflows")
a.start_create("mainworkflow")

Task 1 : wf_NUODATA_TEST depends on None
CREATING Main WORKFLOW: wf_NUODATA_TEST which depends on None
wf_NUODATA_TEST  root task is successfully created
Task 2 : Wkl_NUODATA_TEST_Pre_Load depends on wf_NUODATA_TEST
Task 3 : Wkl_NUODATA_TEST_Pre_Load_LOC_4 depends on None
CREATING Main WORKFLOW: Wkl_NUODATA_TEST_Pre_Load_LOC_4 which depends on None
Wkl_NUODATA_TEST_Pre_Load_LOC_4  root task is successfully created
Task 4 : Wkl_NUODATA_TEST_Pre_Load_LOC_2 depends on Wkl_NUODATA_TEST_Pre_Load_LOC_4
Task 5 : s_NUODATA_USER_TEST depends on None
CREATING Main WORKFLOW: s_NUODATA_USER_TEST which depends on None
s_NUODATA_USER_TEST  root task is successfully created
Task 6 : s_NUODATA_DEPARTMENT_TEST depends on s_NUODATA_USER_TEST
CREATING session : s_NUODATA_DEPARTMENT_TEST which depends on s_NUODATA_USER_TEST
Task 7 : s_NUODATA_SUMMARY_TEST depends on s_NUODATA_DEPARTMENT_TEST
CREATING session : s_NUODATA_SUMMARY_TEST which depends on s_NUODATA_DEPARTMENT_TEST
CREATING SUB WORKFLOW: Wkl_NUO